<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics — Book 1
## Instructor Solutions Manual
### Example Answers: Chapters 3–9 Reflection Cells

---

**For instructor use only.** This notebook contains worked example answers for every reflection cell in the Chapter 3–9 companion notebooks. These are model answers, not the only correct answers — analytical judgement questions may have multiple defensible responses. Where a question asks for a number (e.g. number of alerts), the answer shown is based on the Northgate dataset generated with `random_seed=42`.

**How to use this notebook:**  
Read each chapter section heading. Each reflection prompt is reproduced in full, followed by an example answer in the answer block below it. Students who arrive at materially different but analytically sound conclusions should receive full credit.

---

---
# Chapter 3 — The Transaction Monitoring Lifecycle
## Exercise 3.1: Tracing the Northgate Scenario Through the TM Lifecycle

### Stage 1 — Data Collection

**Prompt:** What data is used? What are the data quality risks?

**Example answer:**

Stage 1 uses transaction data (`nb_transactions.csv`), customer records (`nb_customers.csv`), counterparty reference data (`nb_counterparties.csv`), and account metadata (`nb_accounts.csv`). The Northgate dataset has 23,188 transactions across 500 accounts for the full calendar year 2023.

The primary data quality risk at Stage 1 is **missing or stale customer attribute data**. The mule accounts have `occupation` fields that may be null — a blank occupation is not simply missing data; it may indicate an incomplete onboarding process or a deliberate omission by the customer. For the structuring scenario, the cash-to-income ratio is a critical red flag: a customer depositing USD 85,000 per year in sub-threshold cash against a stated income of USD 20,000 is a 4× anomaly. If `stated_income_usd` is stale (recorded at account opening years earlier), this ratio will be understated and the account may not be flagged. The data quality check cell in the notebook shows that 5% of customer records have missing occupation fields.

### Stage 2 — Scenario Development

**Prompt:** What data is used? What decisions/outputs are produced?

**Example answer:**

Stage 2 uses the transaction dataset, customer attributes, and the regulatory red-flag inventory (FFIEC BSA/AML Manual) to design the detection logic. The output is a scenario specification: **NRB-STRUCT-001**, which monitors rolling 30-day cash deposits below USD 10,000 with a threshold of USD 7,500 and a minimum of 3 transactions.

The key decision at Stage 2 is **threshold calibration**: what rolling sum and transaction count should trigger an alert? This requires knowledge of the population's normal cash behaviour. In the Northgate dataset, legitimate retail customers have average monthly cash-in of USD 967; the mule accounts average USD 48,473. A threshold of USD 7,500 creates clear separation between the two populations. Setting the threshold too low (e.g. USD 3,000) would generate hundreds of alerts from legitimate customers; setting it too high (e.g. USD 15,000) would allow mule accounts to operate undetected.

### Stage 3 — Alert Generation

**Prompt:** What data is used? What does the TMS output?

**Example answer:**

Stage 3 applies the NRB-STRUCT-001 rule logic to the transaction dataset using a rolling time-window aggregation. The output is an alert list: accounts where the 30-day rolling cash deposit sum exceeds USD 7,500 across at least 3 transactions. At default parameters, this produces **47 alerts** from the 500-account Northgate population. All six mule accounts are present; the remaining 41 are false positives — legitimate customers whose cash behaviour happens to breach the threshold.

The transaction counts and rolling sums for each alert are the outputs; they constitute the primary evidence for the investigator at Stage 4. The alert includes the account ID, peak rolling sum, and peak transaction count, but not the underlying reason for the cash activity — that must be gathered by the investigator.

### Stage 4 — Alert Review (L1)

**Prompt:** What data is used? What decision/output is produced?

**Example answer:**

Stage 4 uses the alert record (peak rolling sum, transaction count), the customer profile (occupation, stated income, CRR score, account opening date), and the transaction history (individual deposits, dates, counterparty IDs). The L1 investigator makes a **triage decision**: close the alert as no action required, escalate to L2 for deeper investigation, or place the alert in hibernation pending additional information.

For the Northgate mule accounts, the L1 investigator should note: (a) cash-in amounts clustering in the USD 7,800–9,800 range (sub-threshold structuring pattern), (b) counterparties in high-risk jurisdictions, (c) stated income of USD 18,000–24,000 against annual cash deposits exceeding USD 80,000. These observations are sufficient to escalate — but only if the investigator has access to the counterparty country data and has been trained to recognise the structuring typology.

### Stage 5 — Case Investigation (L2/L3)

**Prompt:** What data is used? What decision/output is produced?

**Example answer:**

Stage 5 uses the full case file assembled at Stage 4, supplemented by **external data** — Companies House records (if the customer is a business), open-source intelligence on counterparties, and formal information requests if warranted. The L2 investigator may also review linked accounts (the six mule accounts share counterparties, which network analysis would surface).

The L2 output is a documented investigation conclusion with supporting evidence: either a recommendation to close the case with rationale, or a recommendation to file a SAR. The quality of this decision depends heavily on the data available — if counterparty country data is absent from the alert feed, the investigator cannot see the geographic risk component and may close a case that should have been filed.

### Stage 6 — SAR Filing or Closure

**Prompt:** What data is used? What decision/output is produced?

**Example answer:**

Stage 6 produces either a **Suspicious Activity Report** (FinCEN format in the US, SARs/STRs in UK/EU) or a case closure with documented rationale. The SAR narrative must describe the suspicious behaviour, the data used to identify it, and the basis for the filing decision. For the Northgate structuring scenario, the SAR would describe the rolling sub-threshold cash deposits, the cash-to-income ratio anomaly, and the counterparty high-risk jurisdiction exposure.

Filing a SAR does not constitute a finding of guilt. It is a report of suspicious activity; law enforcement determines whether to investigate. The tipping-off prohibition (s.333A POCA 2002 in the UK) means the bank cannot inform the customer that a SAR has been filed.

### Synthesis — Which Stage Is Most at Risk?

**Prompt:** Based on Section 2 observations, which stage is most at risk from data quality gaps?

**Example answer:**

**Stage 4 (L1 Alert Review)** is most at risk from the data quality gaps visible in the Northgate dataset. The notebook's data quality check shows that 5% of customer records have missing occupation fields, and `stated_income_usd` may be significantly out of date. An L1 investigator relying on a customer profile that shows occupation = `NULL` and stated income from three years ago cannot make an informed triage decision — they have no baseline against which to assess whether the cash activity is anomalous for this customer's profile.

The second most at-risk stage is **Stage 1 (Data Collection)**: if counterparty country codes are missing or incorrectly mapped, Rule NRB-GEO-003 will not generate alerts on accounts transacting with high-risk jurisdictions, and that dimension of risk will be invisible to the TM system entirely.

---
# Chapter 4 — Rule-Based Transaction Monitoring
## Exercise 4.1: Threshold Sensitivity Analysis

### Question 1 — Understanding the Rule Output

**Prompt:** How many accounts triggered Rule NRB-STRUCT-001? Are all six mule accounts present? Which had the highest peak rolling sum?

**Example answer:**

At the default threshold of USD 7,500 with a minimum of 3 transactions, Rule NRB-STRUCT-001 alerts on **47 accounts**. All six mule accounts (ACC0001–ACC0006) are present in the alert list. The mule accounts have peak 30-day rolling sums in the USD 85,000–110,000 range — roughly 12–15× the alert threshold. The non-mule accounts that trigger the rule have peak rolling sums in the USD 7,500–12,000 range, clustered near the threshold.

The peak transaction count for mule accounts is 3–8 per 30-day window, reflecting the 3–8 cash deposits per month embedded in the dataset construction. The combination of high rolling sum and moderate transaction count is the structuring signature: each deposit is sub-threshold individually, but the aggregate is far above what a legitimate customer's cash-in pattern would produce in a rolling month.

### Question 2 — Threshold Sensitivity

**Prompt:** At what threshold does the rule first fail to recall all six mule accounts? At USD 7,500, how many false positives are there? Which threshold would you recommend for a team that can review 20 alerts per month?

**Example answer:**

The threshold sensitivity analysis shows that all six mule accounts are recalled for every threshold up to approximately USD 12,000. At USD 15,000, the rule begins to miss some mule accounts whose monthly cash-in accumulation falls below this level (the lowest-activity mule accounts average around USD 55,000–65,000 in rolling 30-day sums, but the worst-case window may fall below USD 15,000). 

At the default threshold of USD 7,500, there are **41 false positives** (47 total alerts minus 6 mule accounts). This is a false positive rate of 87% — typical for first-generation rule-based monitoring without segmentation.

For a team with capacity for 20 alerts per month, a threshold of approximately USD 9,000 reduces total alerts to around 15–18 while still capturing all six mule accounts. This is a reasonable starting calibration point; Chapter 6's segmentation will allow further refinement by applying different thresholds per customer segment.

### Question 3 — Limitations of Rule-Based Detection

**Prompt:** Name two types of structuring behaviour that Rule NRB-STRUCT-001 would NOT detect.

**Example answer:**

1. **Multi-account structuring through a single customer**: Rule NRB-STRUCT-001 operates at the individual account level. If a customer operates three linked accounts and deposits USD 3,000 in each per day, no single account breaches the threshold — but the combined daily cash activity is USD 9,000. Entity consolidation across linked accounts is required to detect this pattern, and is beyond the scope of a single-account rolling rule.

2. **Deposits below the minimum transaction count**: If a customer deposits USD 9,500 exactly once in a 30-day window, the rule does not trigger (minimum 3 transactions required). A single large sub-threshold deposit may be a precursor to structuring behaviour or may represent the customer testing the bank's monitoring sensitivity. Neither a single-deposit rule (too many false positives) nor the rolling-sum rule alone addresses this variant.

A third limitation worth noting: the rule treats all transaction types identically within the `CASH_IN` category. It does not distinguish deposits made at the same branch on the same day (a strong structuring indicator) from deposits spread across different weeks. Temporal clustering within the window is not captured.

---
# Chapter 5 — Entity Resolution
## Exercise 5.1: Shared Counterparty Network

### Question 1 — Building the Bipartite Graph

**Prompt:** Using the Northgate transaction data, construct a bipartite graph connecting accounts to the counterparties they transact with. Filter to the 30 high-risk counterparties (CPT0001–CPT0030). How many connected components does the high-risk sub-graph contain? Which accounts share the most counterparties with the six mule accounts?

**Example answer:**

The bipartite graph has **500 account nodes** and **300 counterparty nodes**. After filtering to transactions involving the 30 high-risk counterparties (CPT0001–CPT0030), the mule accounts (ACC0001–ACC0006) are fully contained within a single dense connected component. Each mule account transacts exclusively with high-risk counterparties; their combined counterparty sets overlap heavily — on average, each pair of mule accounts shares 18–24 common counterparties.

In the full graph (all 300 counterparties), the mule accounts form a tight cluster with an internal edge density approximately 4× that of the background population. This structural isolation is a key entity resolution signal: accounts that share high-risk counterparties while having few legitimate-counterparty connections are strong candidates for co-investigation.

The practical implication is that alerting on ACC0001 in isolation (as Rule 1 does) misses the network context entirely. Entity resolution surfaces the remaining five linked accounts before a single SAR is filed, enabling a coordinated case rather than six disconnected reviews.

### Question 2 — Degree Centrality and Hub Accounts

**Prompt:** Compute the degree centrality of each account in the high-risk counterparty sub-graph (an edge exists between an account and a counterparty where at least one transaction occurred). Rank the top 10 accounts by degree. Are all six mule accounts in the top 10? What does a high-degree account represent in an AML context?

**Example answer:**

All six mule accounts appear in the top 10 by degree centrality in the high-risk sub-graph. Their degree scores are substantially higher than any non-mule account because the dataset routes all mule transactions exclusively through the 30 high-risk counterparties. The top 6 accounts by degree are ACC0001–ACC0006 in some order, with degree scores of 20–28 distinct high-risk counterparties each.

A high-degree account in this sub-graph represents a customer with broad exposure across multiple high-risk jurisdictions — a pattern consistent with either a money service business operating without proper licensing, a trade-based money laundering scheme routing payments through multiple correspondent jurisdictions, or a structured cash network using different counterparty shell companies to fragment traceability.

Non-mule accounts in positions 7–10 typically have 1–3 incidental high-risk counterparty transactions. These are likely false positives driven by legitimate trade with counterparties that happen to be domiciled in high-risk jurisdictions — context that a Level 2 investigator would resolve by reviewing the counterparty's business purpose.

### Question 3 — Entity Deduplication via Profile Similarity

**Prompt:** Using the customer profile data (stated income, CRR score, account open date), compute pairwise cosine similarity between the six mule accounts. Are they distinguishable by profile alone? What does this tell you about the limitations of identity-based entity resolution in AML investigations?

**Example answer:**

The six mule accounts have similar but not identical profiles: stated incomes cluster in the USD 18,000–24,000 range, CRR scores are predominantly 3 or 4, and account open dates are spread across 2020–2023. Pairwise cosine similarity values among the six range from approximately 0.87 to 0.97, indicating high profile similarity — but not identity.

This illustrates a fundamental limitation of profile-based entity resolution: in a synthetic dataset, profiles are generated from the same distribution but are not exact duplicates. In real investigations, this corresponds to the scenario where a beneficial owner opens multiple accounts under slightly different spellings, different addresses, or across subsidiary entities — similar enough to warrant investigation but not identical enough to trigger automated deduplication rules.

The practical lesson: profile similarity alone is insufficient for entity resolution in AML. Network structure (shared counterparties, overlapping transaction timing) provides the stronger signal. Combining both — profile similarity as a first filter, network adjacency as the confirming criterion — is the standard ER approach in financial crime analytics.

### Question 4 — ER-Enhanced Alert Prioritisation

**Prompt:** In Chapter 4, Rule NRB-STRUCT-001 produced 47 alerts. Using the entity graph from Exercises 5.1–5.3, how many of those 47 alerts are linked to the mule network (sharing at least one high-risk counterparty with any of the six mule accounts)? How should an analyst use this linkage information to prioritise a queue of 47 alerts?

**Example answer:**

After joining the 47 alerted accounts to the high-risk counterparty sub-graph, typically **8–12 accounts** share at least one high-risk counterparty with the mule cluster — the six mule accounts themselves plus 2–6 non-mule accounts with incidental high-risk counterparty transactions. The exact count depends on the ER linkage threshold (one vs. two shared counterparties).

For alert prioritisation, the 47-alert queue should be treated as three tiers:

1. **Tier 1 — Coordinated case (6 accounts):** The mule network members with dense high-risk counterparty overlap. These should be escalated together to a Level 2 investigator as a single case file, not reviewed individually. Consolidating them surfaces the structuring network rather than treating each account as an isolated alert.

2. **Tier 2 — Network-adjacent (2–6 accounts):** Accounts with incidental high-risk counterparty overlap. Reviewed individually with the counterparty country context displayed; most will resolve as false positives with a legitimate business explanation.

3. **Tier 3 — Structuring pattern only (~35 accounts):** Accounts with no high-risk counterparty linkage. Reviewed on the structuring pattern alone; these have the lowest prior probability of genuine ML given the absence of jurisdictional risk.

This three-tier prioritisation is the operational value of entity resolution: it transforms a flat list of 47 independent alerts into a structured investigation queue, reducing the effective review burden by consolidating the highest-priority cases.

---
# Chapter 6 — Segmentation
## Exercise 6.1: K-Means Segmentation

### Question 1 — Segment Discovery

**Prompt:** Which segment contains all six mule accounts? What features distinguish it? What does unsupervised discovery mean?

**Example answer:**

With K=3, all six mule accounts fall into **Segment 2** (the exact integer label depends on random initialisation — the key is that all six fall into the same segment). This segment is characterised by:

- **Average monthly cash-in**: USD 48,473 — approximately 50× the population average of USD 967
- **Transaction frequency**: ~66 transactions per year — moderate (not the highest-frequency segment)
- **Cash ratio**: ~1.0 (essentially all transactions are cash deposits, whereas legitimate customers average 15% cash-in)

The other two segments are: a large low-activity segment (~235 customers, average monthly cash-in USD 967, low cash ratio) representing the bulk of standard retail customers; and a moderate-activity segment (~259 customers, average monthly cash-in USD 1,200, mixed transaction types).

The fact that K-Means discovered this cluster **without being told which accounts were mules** is significant: it demonstrates that the mule accounts' behaviour is genuinely distinctive in the feature space. An algorithm with no domain knowledge identified the six accounts as an anomalous group purely from their transaction statistics. This validates the feature design choices (cash-in amount, frequency, ratio) as effective discriminators of the structuring behaviour.

### Question 2 — Choosing K

**Prompt:** Does the elbow chart support K=3? Would K=3 be appropriate for 500,000 customers?

**Example answer:**

The elbow chart for the Northgate dataset shows a clear reduction in inertia between K=2 and K=3, with diminishing returns from K=4 onwards. The "elbow" — the point where increasing K produces proportionally smaller inertia reductions — is visible at K=3, supporting our choice. The gap between K=3 and K=4 inertia is smaller than the gap between K=2 and K=3, confirming K=3 as a defensible selection for this dataset.

For a real bank with 500,000 customers, K=3 would almost certainly be too coarse. A population of this size would contain many distinct customer types: high-net-worth individuals, small businesses, students, expatriates, pensioners, and so on. Using K=3 would aggregate genuinely different populations into the same segment and apply the same monitoring threshold to customers with very different normal behaviours. In practice, retail banks typically use 8–20 segments for consumer accounts, with separate segmentation frameworks for commercial and private banking populations. The elbow method must be rerun on the full population, and domain expert input is required to validate that the resulting segments are operationally meaningful.

### Question 3 — Segment Labelling

**Prompt:** Label each segment and explain which threshold should be tightened.

**Example answer:**

Based on the enriched segment profile (with CRR score and stated income):

**Segment 0 — Standard Retail Customer:** Low average cash-in (~USD 800/month), low transaction frequency, low cash ratio (~12%), CRR score 2.1, stated income ~USD 22,000. These are ordinary retail depositors whose cash activity is modest and predictable. Monitoring threshold: standard (USD 7,500 rolling 30-day is appropriate).

**Segment 1 — Moderate Transactor:** Moderate cash-in (~USD 1,200/month), higher frequency, mixed transaction types including cards and transfers. CRR score 2.4, stated income ~USD 28,000. Likely self-employed or small business-adjacent customers. Monitoring threshold: slightly elevated (USD 9,000–12,000) to reduce false positives from legitimate cash-heavy businesses.

**Segment 2 — High-Cash Structured Risk (Mule Cluster):** Very high average monthly cash-in (~USD 48,000), high cash ratio (~100%), moderate frequency, CRR score 3.2. This segment contains all six mule accounts. The threshold for this segment should be **tightened** significantly — or an additional rule (minimum transaction frequency check) should be applied specifically to this segment, since even a USD 5,000 rolling sum for a customer in this cluster is anomalous relative to their peers.

---
# Chapter 7 — Tuning and Calibration
## Exercise 7.1: Rule Responsiveness and Multi-Rule Overlap

### Question 1 — Rule 2 Behaviour

**Prompt:** How many accounts triggered NRB-VEL-002? How many mule accounts? Describe the detected pattern in plain English.

**Example answer:**

At default settings (min_txns=5, window_days=14, max_gap_days=3), Rule NRB-VEL-002 alerts on **357 accounts** — a substantially larger pool than Rule 1's 47. All six mule accounts are captured. The high alert count reflects that many ordinary retail customers transact frequently (5 transactions in 14 days is not unusual for an active current account), making this rule prone to false positives at the default threshold.

Rule NRB-VEL-002 detects accounts making rapid-fire transactions in a short window, where consecutive transactions are separated by no more than 3 days. In plain English: it flags accounts that appear to be "running" transactions — making multiple deposits or payments in quick succession, as if urgently moving money through the account. For a mule account, this reflects the operational reality of receiving cash from handlers and rapidly structuring it into deposits. For a legitimate customer, it might reflect bill payment days, salary receipt cycles, or active use of a current account for daily spending.

### Question 2 — Tuning Rule 2

**Prompt:** At what min_txns does Rule 2 first miss a mule account? What is the false positive rate at min_txns=5? Which setting would you choose for a bank with limited investigation capacity?

**Example answer:**

The responsiveness analysis shows that Rule 2 captures all six mule accounts across all tested values of min_txns (3 through 8). The mule accounts have sufficiently high transaction frequency — 36–96 transactions per year, distributed across rapid-burst episodes — that even the most restrictive setting (min_txns=8) does not miss them in the 14-day window.

At the default setting of min_txns=5, there are **351 false positives** (357 total minus 6 mule accounts) — a false positive rate of 98.3%. This is extremely high and reflects that NRB-VEL-002 at default settings should not be used as a standalone alert generator; it is most useful as an additive flag in a multi-rule system.

For a bank with limited investigation capacity, I would recommend **min_txns=7** or **min_txns=8**: this significantly reduces the alert volume (likely to below 100) while still capturing all mule accounts. Alternatively, the rule should be used only as a secondary flag — accounts that trigger BOTH Rule 1 and Rule 2 are the primary investigation queue, while accounts triggering only Rule 2 are placed in a lower-priority queue or monitored for 30 days before escalation.

### Question 3 — Multi-Rule Priority

**Prompt:** How many accounts triggered both rules? Are all six mule accounts in the "both" group? How would you communicate the HIGH/MEDIUM distinction to an L1 investigator?

**Example answer:**

At default settings, **46 accounts** trigger both Rule 1 (Structuring) and Rule 2 (Velocity) simultaneously. All six mule accounts are among them. The dual-trigger accounts are statistically less likely to be false positives than single-trigger accounts: any given retail customer may have a high cash month (triggering Rule 1) or a flurry of transactions around a bill payment cycle (triggering Rule 2), but exhibiting both patterns simultaneously is less likely by chance.

To communicate the HIGH/MEDIUM distinction to an L1 investigator: *"HIGH priority alerts have triggered two or more detection rules simultaneously. Investigate these first. MEDIUM priority alerts have triggered one rule — they remain important, but your queue should prioritise the HIGH cases. For a HIGH alert, you should be able to explain in your case notes why the account triggered both rules and whether the two patterns are related (e.g., is the velocity burst associated with the cash deposit dates?)."*

This framing gives the investigator a clear action (prioritise HIGHs) and a concrete documentation requirement (explain both rule triggers), without requiring them to understand the underlying algorithm.

---
# Chapter 8 — Risk and Coverage Assessments
## Exercise 8.1: Three-Rule Coverage Analysis

### Question 1 — Rule 3 Performance

**Prompt:** How many accounts triggered NRB-GEO-003? Are all six mule accounts captured? Which country code appears most frequently?

**Example answer:**

At default settings (min_txns=2, min_amount=5,000), Rule NRB-GEO-003 alerts on **265 accounts**. All six mule accounts are captured — they were designed to transact exclusively with counterparties in high-risk jurisdictions (counterparty IDs CPT0001–CPT0030, mapped to the seven FATF high-risk country codes). The mule accounts have hr_txn_count of 36–96 per year against the minimum threshold of 2, so they are captured easily at any reasonable parameter setting.

Among the 265 accounts, some are non-mule customers who happen to have transacted with counterparties in high-risk jurisdictions. In a real bank, this might represent legitimate international trade, remittances to family members in listed countries, or business relationships with counterparties in jurisdictions that are listed for AML/CFT reasons but where legitimate commerce continues.

The most frequently appearing high-risk country code in the dataset is **AF** (Afghanistan) or **KP** (North Korea) depending on random seed — both appear among the 30 high-risk counterparties embedded in the dataset. The exact distribution can be read from the country breakdown chart in the notebook.

### Question 2 — Three-Rule Coverage Matrix

**Prompt:** How many mule accounts are caught by all three rules? Are any caught by only one? What would you do with tri-rule vs single-rule accounts?

**Example answer:**

All six mule accounts are captured by all three rules simultaneously: they structure cash (Rule 1), transact in rapid bursts (Rule 2), and use high-risk country counterparties (Rule 3). This is a consequence of the dataset design — each mule account was constructed with all three red-flag behaviours. In a real dataset, some mule accounts might exhibit only one or two of these patterns, depending on the specific money-laundering methodology being employed.

For investigation prioritisation:

**Tri-rule accounts** (all three rules triggered): Highest priority queue. Three independent red flags pointing at the same account represents a compelling pattern of concern. These accounts should be investigated by L2 from the outset, not routed through L1 triage.

**Dual-rule accounts**: High priority. Assign to experienced L1 investigators with a documentation requirement to explain both rule triggers.

**Single-rule accounts**: Medium priority. Standard L1 review. If Rule 1 only — assess cash pattern against income. If Rule 2 only — assess velocity pattern against account type and normal behaviour. If Rule 3 only — assess counterparty risk in context of the customer's stated business activities.

### Question 3 — Coverage Gaps

**Prompt:** List two FFIEC typologies not covered by the three-rule system. Propose a fourth rule for each gap. How would you present coverage gaps to a compliance committee?

**Example answer:**

**Gap 1 — Round-dollar transaction patterns:** The FFIEC manual lists accounts that conduct transactions in round amounts (e.g. USD 5,000.00, USD 10,000.00 exactly) as a structuring red flag. None of the three rules detect this pattern — Rule 1 monitors total rolling sum, not individual transaction round-amounts. A fourth rule could be: *"Flag accounts where more than 40% of cash deposits in a calendar month are exact round-hundreds amounts."*

**Gap 2 — Multiple individuals depositing into a single account:** The FFIEC manual notes that deposits made by multiple different individuals on behalf of the same account — often called "smurfing" — is a structuring variant the rules cannot detect because the deposit data does not include the name of the physical person making the deposit, only the account receiving it. A fourth rule would require teller-captured depositor identity data linked to the transaction record — an infrastructure requirement, not just a rule-writing exercise.

To present coverage gaps to a compliance committee: frame each gap with (1) the regulatory reference (FFIEC section and page), (2) the consequence of the gap (what real-world activity it would miss), (3) the volume/value assessment of that activity in the existing transaction population where estimable, and (4) the remediation options — a new rule, a data infrastructure project, or a documented acceptance of residual risk. The committee needs to make a decision; give them the evidence to do so.

---
# Chapter 9 — Alert Triage and Machine Learning
## Exercise 9.1: Isolation Forest Anomaly Scoring

### Question 1 — Isolation Forest Results

**Prompt:** What were the ranks of the six mule accounts? Were all in the top 10? Explain what the anomaly score means to a non-technical compliance manager.

**Example answer:**

In the baseline Isolation Forest run (contamination=0.05, random_seed=42), the six mule accounts rank as follows:

| Account | Rank | IF Score |
|---------|------|----------|
| ACC0003 | 1 | −0.767 |
| ACC0005 | 2 | −0.756 |
| ACC0002 | 3 | −0.736 |
| ACC0001 | 4 | −0.732 |
| ACC0004 | 5 | −0.724 |
| ACC0006 | 6 | −0.720 |

All six mule accounts rank in positions 1–6, ahead of all 386 other alerted accounts. The model successfully identifies the six known suspicious accounts as the most anomalous in the alert pool.

**Explanation for a compliance manager:** *"The Isolation Forest score tells you how unusual an account is compared to all the other accounts that triggered our detection rules. Think of it as a second opinion on the alert queue. A very low (negative) score means the account is genuinely different from the rest — it stands out in multiple ways at once. A score closer to zero means the account is unusual enough to be in the alert pool but doesn't stand out as dramatically. We use this score to tell our investigators where to look first: start at the top of the ranked list and work down. The score does not tell them what the account is doing — that's for the investigator to determine. But it tells them which account is most likely worth their time."*

### Question 2 — Contamination Parameter

**Prompt:** How stable are mule ranks across contamination values 0.01–0.20? What contamination would you set if 3% of alerted accounts are genuinely suspicious?

**Example answer:**

The contamination sensitivity analysis shows that mule account ranks are **relatively stable** across the tested range. The average mule rank remains in the 1–6 range for all contamination values tested, with some variation in the exact ordering among the six mule accounts. The model's ability to identify the mule accounts as anomalous is robust to the contamination parameter choice because the mules' feature values are so extreme relative to the rest of the alert pool.

If your bank estimates that 3% of alerted accounts represent genuine suspicious activity, set `contamination=0.03`. This tells the Isolation Forest that approximately 3% of the alert pool are anomalies — consistent with a bank's empirical SAR filing rate relative to total alerts reviewed. Note that `contamination` does not change which accounts are flagged as anomalous in an absolute sense; it affects the threshold used by `fit_predict()` to classify accounts as anomalous vs normal. The `score_samples()` output — which we use for ranking — is independent of the contamination parameter. Using `score_samples()` for ranking (as the notebook does) means the contamination parameter has minimal effect on investigation prioritisation.

Disclosing the contamination parameter to regulators: yes, it should be documented in the model card as a design choice. The documentation should explain: (a) what the parameter represents, (b) how the value was selected, and (c) that the ranking is based on `score_samples()` which is independent of this parameter.

### Question 3 — Feature Importance and Model Governance

**Prompt:** Which two features have highest permutation importance? Does low importance for rule1_flag mean Rule 1 is useless? List three model validation requirements from SR 11-7.

**Example answer:**

The permutation importance analysis typically shows that **`total_cash_in`** and **`hr_ratio`** have the highest importance — shuffling either of these features most degrades the mule accounts' anomaly scores. This makes intuitive sense: the mule accounts' most extreme feature values are their very high total cash deposits and their near-100% high-risk counterparty exposure. Shuffling these features breaks the combination that makes the mule accounts stand out.

If `rule1_flag` shows low permutation importance: this does **not** mean NRB-STRUCT-001 is a poor rule. Rule 1's information is already captured, more precisely, by the continuous feature `total_cash_in` — accounts that trigger Rule 1 have high total_cash_in values. The binary flag adds little marginal information once the continuous feature is already in the model. This is a case of feature redundancy, not rule redundancy. NRB-STRUCT-001 remains the primary legal and regulatory foundation for generating the alert pool; the Isolation Forest feature simply captures the same underlying information in a richer continuous form.

Three SR 11-7 requirements a model validator would expect to see for this model:
1. **Conceptual soundness documentation**: an explanation of why Isolation Forest is appropriate for this use case, including assumptions about anomaly distribution and the implications of unsupervised learning without labelled data.
2. **Performance testing**: evidence that the model produces better alert prioritisation than a simple rule-based ranking (e.g., comparison of mule recall in the top N% of ranked accounts vs random ranking).
3. **Sensitivity analysis**: evidence that the model's rankings are stable across reasonable variations in hyperparameters (contamination, n_estimators) and across different random seeds.

### Question 4 — System Reflection

**Prompt:** Describe the end-to-end system in 3–4 sentences. What is the biggest remaining weakness? What proportion of real suspicious accounts show multiple red flags?

**Example answer:**

**End-to-end system description:** The Northgate TM system begins by applying three rule-based filters to the raw transaction dataset — NRB-STRUCT-001 (rolling cash structuring), NRB-VEL-002 (rapid-fire velocity), and NRB-GEO-003 (high-risk country counterparty exposure) — producing a combined alert pool of 392 accounts. An Isolation Forest model then scores every account in the alert pool using seven features (three continuous behavioural metrics plus three rule flags), ranking them from most to least anomalous without requiring any labelled training data. Investigators work through the ranked queue, starting from the top; the six mule accounts embedded in the synthetic dataset rank in positions 1–6 by anomaly score. The interactive investigation dashboard (Section 4) provides a structured case review interface for each alerted account, including the transaction history, monthly cash-in chart, rule flags, and a disposition recording mechanism.

**Biggest remaining weakness:** The system operates at the **individual account level**. It can identify that ACC0001 is suspicious, but it cannot see that ACC0001, ACC0002, and ACC0003 are part of the same coordinated network, sending funds to the same ultimate beneficiary. Network-level detection — graph analysis to surface connected components, shared counterparties, and fund-flow paths — is beyond the scope of the current system. In a real mule network, the individual accounts are designed to look independent; only network analysis reveals the coordination.

**Multiple red flags in real suspicious accounts:** Empirical studies from SAR analysis suggest that approximately 60–70% of confirmed SAR cases involve two or more red flags from a bank's scenario inventory. Accounts that trigger only a single rule are more likely to be false positives. This validates the multi-rule triage approach: the three-rule system and the dual/tri-trigger priority queue are a reasonable approximation of the multi-red-flag reality, even without network analysis.

---
# Notes for Instructors

## Grading guidance

Questions that ask for a specific number (e.g. alert counts, mule ranks) have definitive answers based on the fixed random seed (42) in the Northgate dataset generator. These can be auto-graded or marked by comparison.

Questions that ask for analytical judgement (e.g. "which threshold would you recommend?", "what is the biggest weakness?") should be marked on the quality of reasoning, not the specific conclusion. A student who recommends min_txns=6 with a well-reasoned argument about investigator capacity deserves full credit even if the model answer recommends min_txns=7.

Questions that ask for written explanations ("describe in plain English", "explain to a compliance manager") should be marked on clarity, accuracy, and appropriate register — avoiding jargon where the target audience is a non-technical reader, using precise technical language where the target audience is a model validator.

## Common errors to watch for

- Confusing **precision** and **recall** in the TM context (precision = % of alerts that are genuine; recall = % of genuine cases that generated an alert)
- Treating a low Isolation Forest anomaly score as equivalent to a confirmed finding — the score is a prioritisation signal, not a determination
- Forgetting that the contamination parameter does not affect `score_samples()` output, only `fit_predict()` labels
- Conflating **coverage** (does a scenario exist for this typology?) with **calibration** (is the scenario's threshold set correctly to detect instances of that typology?)

---
*© Compliance Analytics Ltd. For instructor use only. Not for distribution to students.*